#### THIRD LAYER
##### GOLD LAYER
nb_customer_silver_gold

In [0]:
%sql
WITH latest_customer_silver AS (
    SELECT *
    FROM intellibi_catalog.intellibi_silver.cleanedCustomer
    -- WHERE ingest_ts >
    -- (
    --     SELECT COALESCE(MAX(start_effective_date), TO_TIMESTAMP('1900-01-01'))
    --     FROM intellibi_catalog.intellibi_gold.refinedCustomer
    -- )
),
silver_gold_rec AS (
    SELECT
        g.customer_sk,
        s.customer_id,
        s.customer_name,
        s.email,
        s.phone,
        s.address,
        s.city,
        s.state,
        s.country,
        s.zip_code,
        s.segment,
        s.ingest_ts,
        g.is_active,
        g.rec_version,
        g.start_effective_date,
        g.end_effective_date,
        CASE
            WHEN g.customer_sk IS NULL THEN 1
            ELSE g.rec_version + 1
        END AS new_rec_version,
        CASE
            WHEN g.customer_sk IS NULL THEN 'NEW'
            WHEN
                s.customer_name <> g.customer_name OR
                s.email <> g.email OR
                s.phone <> g.phone OR
                s.address <> g.address OR
                s.city <> g.city OR
                s.state <> g.state OR
                s.country <> g.country OR
                s.zip_code <> g.zip_code OR
                s.segment <> g.segment
            THEN 'CHANGE'
            ELSE 'NO_CHANGE'
        END AS rec_flag
    FROM latest_customer_silver s
    LEFT JOIN intellibi_catalog.intellibi_gold.refinedCustomer g
        ON s.customer_id = g.customer_id AND g.is_active = 'Y'
),
insert_flag AS (
    -- Insert new record for NEW or CHANGE
    SELECT 
        NULL AS cust_merge_key,
        customer_id, customer_name, email, phone, address, city, state, country, zip_code, segment, ingest_ts,
        'Y' AS is_active,
        new_rec_version AS rec_version,
        CURRENT_TIMESTAMP() AS start_effective_date,
        TO_DATE('9999-12-31','yyyy-MM-dd') AS end_effective_date,
        new_rec_version, rec_flag, 'INSERT' AS merge_flag
    FROM silver_gold_rec
    WHERE rec_flag IN ('NEW', 'CHANGE')
),
changed_flag AS (
    -- Update existing active record to inactive for CHANGE
    SELECT 
        customer_sk AS cust_merge_key,
        customer_id, customer_name, email, phone, address, city, state, country, zip_code, segment, ingest_ts,
        'N' AS is_active,
        rec_version,
        start_effective_date,
        CURRENT_TIMESTAMP() AS end_effective_date,
        rec_version, rec_flag, 'UPDATE' AS merge_flag
    FROM silver_gold_rec
    WHERE rec_flag = 'CHANGE'
)
SELECT * FROM insert_flag
UNION ALL
SELECT * FROM changed_flag;

In [0]:
# %sql
# WITH latest_customer_silver as (
#     SELECT * FROM intellibi_catalog.intellibi_silver.cleanedCustomer
#     WHERE ingest_ts > ( SELECT COALESCE(MAX(start_effective_date), TO_TIMESTAMP('1900-01-01'), 'YYYY-MM-DD') FROM intellibi_catalog.intellibi_gold.refinedCustomer )
# )
# select 
# g.customer_sk as customer_sk,
# s.*,
# g.is_active as is_active,
# g.rec_version as rec_version,
# current_timestamp() as start_effective_date,
# to_date('9999-12-31', 'yyyy-MM-dd') as end_effective_date,
# CASE WHEN g.customer_sk IS NULL THEN 1 ELSE g.rec_version + 1 END as new_rec_version,
# CASE WHEN g.customer_sk IS NULL THEN 'NEW'
#     WHEN s.email <> g.email or s.phone <> g.phone or s.address <> g.address or s.city <> g.city or s.state <> g.state or s.country <> g.country or s.zip_code <> g.zip_code 
#     THEN 'UPDATE'
#     ELSE 'NO_CHANGE' END as change_type
# from latest_customer_silver s
# left join intellibi_catalog.intellibi_gold.refinedCustomer g
# on s.customer_id = g.customer_id;

In [0]:
# %sql
# MERGE INTO
#   intellibi_catalog.intellibi_gold.refinedCustomerAS TGT
# USING (
#   SELECT
#     *
#   FROM
#     intellibi_catalog.intellibi_silver.cleanedCustomer
#   WHERE
#     ingest_ts
#       > (
#         SELECT
#           COALESCE(MAX(start_effective_date), TO_TIMESTAMP('1900-01-01'), 'YYYY-MM-DD')
#         FROM
#           intellibi_catalog.intellibi_gold.refinedCustomer
#       )
# ) AS SRC
# ON
#   TGT.Customer_ID = SRC.Customer_ID
# WHEN MATCHED AND
#   (
#     TGT.Customer_Name <> SRC.Customer_Name
#     OR TGT.Gender <> SRC.Gender
#     OR TGT.Age <> SRC.Age
#     OR TGT.City <> SRC.City
#     OR TGT.custpurPrice <> SRC.custpurPrice
#     OR TGT.purchase_date <> SRC.purchase_date
#   )
#   THEN UPDATE SET
#   TGT.Customer_Name = SRC.Customer_Name,
#   TGT.Gender = SRC.Gender,
#   TGT.Age = SRC.Age,
#   TGT.City = SRC.City,
#   TGT.custpurPrice = SRC.custpurPrice,
#   TGT.purchase_date = SRC.purchase_date,
#   TGT.last_u_ts = current_timestamp()
# WHEN NOT MATCHED THEN INSERT (
#     TGT.Customer_ID,
#     TGT.Customer_Name,
#     TGT.Gender,
#     TGT.Age,
#     TGT.City,
#     TGT.custpurPrice,
#     TGT.purchase_date,
#     TGT.intial_load_ts,
#     TGT.last_u_ts
#   )
#   VALUES (
#     SRC.Customer_ID,
#     SRC.Customer_Name,
#     SRC.Gender,
#     SRC.Age,
#     SRC.City,
#     SRC.custpurPrice,
#     SRC.purchase_date,
#     current_timestamp(),
#     current_timestamp()
#   );

In [0]:
%sql
WITH latest_cust_silver AS (
  SELECT * FROM intellibi_catalog.intellibi_silver.cleanedCustomer-- WHERE ingest_ts >
  -- (
  --     SELECT COALESCE(MAX(start_effective_ts), TO_TIMESTAMP('1990-01-01', 'yyyy-MM-dd'))
  --     FROM intellibi_catalog.intellibi_gold.RefinedCustomer
  -- )
),
silver_gold_rec AS(
SELECT
  s.*,
  g.customer_sk AS customer_sk,
  g.is_active AS is_active,
  g.rec_version AS rec_version,
  CURRENT_TIMESTAMP() AS start_effective_ts,
  TO_DATE('9999-12-31', 'YYYY-MM-DD') AS end_effective_ts,
  CASE
    WHEN g.customer_sk IS NULL THEN 1
    ELSE g.rec_version + 1
  END AS new_rec_version,
  CASE
    WHEN g.customer_sk IS NULL THEN 'NEW'
    WHEN
      s.email <> g.email
      OR s.address <> g.address
      OR s.phone <> g.phone
      OR s.city <> g.city
      OR s.state <> g.state
      OR s.country <> g.country
      OR s.zip_code <> g.zip_code
    THEN
      'CHANGED'
    ELSE 'NO_CHANGE'
  END AS change_type
FROM
  latest_cust_silver s
    LEFT JOIN intellibi_catalog.intellibi_gold.RefinedCustomer g
      ON s.customer_id = g.customer_id
  )    
  select * from silver_gold_rec;

In [0]:
%sql
SELECT * FROM intellibicatalogo.intellibi_gold.refinedMonthlySales;